# Atlas-Guided ViT-UNETR: prototype workflow

End-to-end workflow built **after** `AtlasGuidedUNETR_fixed.ipynb` (the supervised prototype). That notebook
feeds the model its own ground-truth label as the anatomical prior, so its Dice numbers measure a model that is
handed the answer. This notebook replaces that with priors built from 10 reference atlases by registration,
which is what the model would have at prediction time.

| # | Stage | Code | Status |
|---|---|---|---|
| 1 | Data split and leakage control | `data/splits.py` | done |
| 2 | Supervised prototype, distance-map fix, baseline check | `data/transforms.py`, `scripts/evaluate_checkpoint.py` | done |
| 3 | Atlas selection (10 temporary atlases) | `scripts/select_atlas_cases.py` | done |
| 4 | Landmarks | `atlas/landmarks.py` | done |
| 5 | Golden transformation and reliable features | `atlas/golden_transform.py`, `atlas/tps.py`, `atlas/reliable_features.py` | done |
| 6 | Atlas → target registration and label warping | `atlas/matching.py`, `ransac.py`, `registration.py`, `label_warping.py` | done; atlas-only Dice 0.45 to 0.48 on 2 val cases |
| 7 | Priors P(v), D(v), w(v) | `data/transforms.py` (`AtlasPriorsd`) | done |
| 8 | Model and Experiment 1 | `models/`, `training/`, `scripts/train.py` | set up, full run pending on a server |
| 9 | Redesigned atlas-aware model (`new_model_plan.md` A–F) | this notebook, section 9 | prototype, 21/21 checks pass |
| 10 | Full run: register every target, train the redesigned model from scratch | `scripts/register_targets.py`, `scripts/train_prototype_v2.py` | runs with `FULL_RUN = True` (server) |
| – | CT edge and HU losses (Experiments 3–4) | none | **not implemented** |

**Server run.** With `FULL_RUN = True` (configuration cell, the default) running the notebook top to bottom
registers all ~670 targets and trains the redesigned model on the full training split (section 10). That takes
many hours; for a quick local look set `FULL_RUN = False`.

**How to use it.** The cells run top to bottom. Anything slow is behind a `RUN_*` flag in the configuration cell.
With every flag `False` the notebook reads existing artifacts under `cache/` and `experiments/` and only does
cheap computation. Full-scale runs (registering ~670 targets, training 50+ epochs) belong on a server; the exact
commands are in `RUN_GUIDE.md`. Decisions, bugs found and results so far are in `research_audit.md` and
`experiments/registration_log.md`.

**Read the numbers with care.** Every result below comes from a small local chunk (2 to 3 cases), not from the
full validation set. They show that each stage works and roughly how well, not final performance.

Unit tests live in `tests/`; section 9 also checks the redesigned model in place before it is trained.

## 0. Setup

Locates the repository root (so the notebook works from any working directory inside it), prints versions and
the GPU, and defines the run flags.

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore", category=FutureWarning)


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "models" / "segmentation_model.py").exists() and (p / "atlas").is_dir():
            return p
    raise RuntimeError("Open this notebook from inside the repository (models/ and atlas/ were not found).")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import monai
import SimpleITK as sitk
import matplotlib.pyplot as plt
from monai.utils import set_determinism

set_determinism(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("repo      :", REPO_ROOT)
print("torch     :", torch.__version__, "| monai", monai.__version__, "| SimpleITK", sitk.Version_VersionString())
print("device    :", device, "-", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU only")

### Configuration and run flags

| Flag | What it does when `True` | Rough cost (this laptop) |
|---|---|---|
| `RUN_BASELINE_EVAL` | re-runs the baseline checkpoint on `BASELINE_LIMIT` validation cases with ground-truth priors | a few min |
| `RUN_ATLAS_SELECTION` | rescans every training label and rewrites `atlas/atlas_selection.json` (deterministic, same result) | ~3 min |
| `RUN_LIBRARY_BUILD` | rebuilds the reliable-landmark library (it is also built automatically if missing) | ~1 min |
| `RUN_REGISTRATION` | registers the 10 atlases onto `N_REGISTER` validation targets and caches the warped labels | ~3 to 4 min per target |
| `RUN_SINGLE_PAIR` | one atlas onto one target in-process, showing every registration stage | ~30 s |
| `RUN_TRAIN_SMOKE` | a few in-process training steps on cached cases (demonstration, nothing is saved) | a few min |
| `RUN_TRAIN_SCRIPT` | launches a tracked training run through `scripts/train.py` (config, git state, metrics saved) | hours, use a server |
| **`FULL_RUN`** | **switches on the three flags below: the full experiment with the redesigned model (section 10)** | **many hours** |
| `RUN_FULL_REGISTRATION` | registers every train and validation target, `REG_SHARDS` processes in parallel; cached ones are skipped | ~3.6 min per target per process |
| `RUN_V2_BENCH` | times a few training steps and prints the estimated training time | ~2 min |
| `RUN_V2_FULL_TRAIN` | trains the redesigned model from scratch on all registered targets; resumes if interrupted | ~14 min per epoch on 4 GB |
| `RUN_V2_SMOKE` | section 9's three-step demo (off in a full run) | ~2 min |

In [ ]:
from data.splits import DEFAULT_DATA_DIR

DATA_DIR = DEFAULT_DATA_DIR          # ../Dataset060_Merged_Def, override if yours is elsewhere
IMG_SIZE = (96, 96, 96)
NUM_CLASSES = 13                     # background + 12 organs
ORGAN_IDS = list(range(1, NUM_CLASSES))

RUN_BASELINE_EVAL = False
BASELINE_LIMIT = 3
RUN_ATLAS_SELECTION = False
RUN_LIBRARY_BUILD = False
RUN_REGISTRATION = False
N_REGISTER = 2
RUN_SINGLE_PAIR = True
RUN_TRAIN_SMOKE = False
RUN_TRAIN_SCRIPT = False
TRAIN_SCRIPT_ARGS = ["--run-name", "nb_" + time.strftime("%Y%m%d_%H%M%S"), "--epochs", "50", "--amp"]

# Redesigned model (sections 9 and 10). FULL_RUN = True is the server configuration.
FULL_RUN = True
RUN_FULL_REGISTRATION = FULL_RUN
REG_SHARDS = max(1, min(8, (os.cpu_count() or 4) // 4))   # parallel registration processes; lower it if the GPU runs out of memory
RUN_V2_BENCH = FULL_RUN
RUN_V2_FULL_TRAIN = FULL_RUN
RUN_V2_SMOKE = not FULL_RUN
V2_EXPERIMENT = "exp_03_v2_full"
V2_RUN_NAME = "v2_full"
V2_EPOCHS = 100
V2_EFFECTIVE_BATCH = 4                # micro-batch follows GPU memory, the rest is gradient accumulation
V2_NUM_WORKERS = None                 # None: min(8, cpu_count // 2)


def run_script(args):
    # Runs a repo script in a subprocess and streams its output. The heavy, cache-writing steps stay
    # scripts so they can be run unchanged on a server; the notebook only orchestrates and inspects.
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(cmd))
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in proc.stdout:
        if "FutureWarning" in line or "warn_deprecated" in line or line.strip().startswith("warnings.warn"):
            continue
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"{args[0]} exited with code {rc}")

## 1. Data and split

The dataset is 680 paired CT volumes and 12-organ labels from three sources (`amos_*`, `img*`, `s*`). The
train/validation split reproduces the supervised prototype exactly (`random.seed(42)`, shuffle, 80/20), so every
number stays comparable with its epoch-101 result.

Two rules keep the atlas work free of leakage:
- the 10 atlas cases come from the **training** portion and are removed from the training targets, so an atlas
  never gets a supervised loss on the same label that also supplies its prior;
- validation is untouched: the same 136 cases as the baseline.

In [ ]:
from data.splits import train_val_split, split_with_atlases_removed, atlas_names, case_name


def source_of(case):
    return re.match(r"[A-Za-z]+", case_name(case["image"])).group(0)


train_all, val_cases = train_val_split(DATA_DIR)
train_targets, _ = split_with_atlases_removed(DATA_DIR)   # also raises if an atlas is in validation
atlases = atlas_names()

print(f"paired cases      : {len(train_all) + len(val_cases)}")
print(f"train (baseline)  : {len(train_all)}   validation: {len(val_cases)}")
print(f"atlases           : {len(atlases)}   -> train targets: {len(train_targets)}")
print("train targets by source:", dict(Counter(source_of(c) for c in train_targets)))
print("validation by source   :", dict(Counter(source_of(c) for c in val_cases)))
print("first validation cases :", [case_name(c["image"]) for c in val_cases[:3]])

## 2. The supervised prototype, its leakage, and the distance-map fix

In the prototype pipeline (`prior_source="gt"`) the case's own label is one-hot encoded and used as the
probability map P(v). A distance transform of that same label gives D(v). Both go into the model during training
and validation, so the reported Dice (0.6505 at epoch 101) describes a model that can read the answer from its
input. At prediction time on a new scan no label exists, which is the problem the atlas pipeline solves.

In [ ]:
from data.transforms import build_transforms, PRIOR_KEYS

print("which tensors feed the model as (P, D):", PRIOR_KEYS)

tf_gt = build_transforms(IMG_SIZE, NUM_CLASSES, train=False, prior_source="gt", legacy_buggy_distance=True)
sample_gt = tf_gt(val_cases[0])
print({k: tuple(v.shape) for k, v in sample_gt.items() if hasattr(v, "shape")})
print("P(v) for the prototype IS the label: sum over classes =", float(sample_gt["label"].sum(0).mean()),
      "| it is binary:", bool(((sample_gt["label"] == 0) | (sample_gt["label"] == 1)).all()))

### Distance-map bug (found and fixed)

`CustomSpatialDistanceTransformd` called `distance_transform_edt(mask)`. SciPy measures the distance from each
**non-zero** voxel to the nearest zero, so the prototype's D(v) was 0 everywhere outside the organ and grew
towards the organ centre, the opposite of D_c(v) = distance to the organ (0 inside, growing outward). The checkpoint
was trained with this, so `legacy_buggy_distance=True` reproduces it, and all new work uses the fixed default.

In [ ]:
from data.transforms import CustomSpatialDistanceTransformd

size, half, c = 9, 2, 4
block = torch.zeros((3, size, size, size))
block[0] = 1.0
block[0, c - half:c + half + 1, c - half:c + half + 1, c - half:c + half + 1] = 0.0
block[1, c - half:c + half + 1, c - half:c + half + 1, c - half:c + half + 1] = 1.0

def profile(legacy):
    t = CustomSpatialDistanceTransformd(keys=["label"], to_closest_foreground=True, include_background=False,
                                        output_postfix="_distance", legacy_buggy_distance=legacy)
    return t({"label": block})["label_distance"][0][c, c].numpy().round(1)   # line through the organ centre

print("position along a line through the organ (organ occupies indices 2..6)")
print("legacy (buggy):", profile(True))
print("fixed         :", profile(False))

### Baseline reproduction (optional)

`scripts/evaluate_checkpoint.py` runs validation only (the prototype notebook could only resume training). It
needs `--legacy-buggy-distance` for `best_model.pth`. A 3-case run gave Dice 0.6757, consistent with the epoch-101
0.6505; the full 136-case run is meant for a server (see `baseline_reproduction.md`).

In [ ]:
if RUN_BASELINE_EVAL:
    run_script(["scripts/evaluate_checkpoint.py", "--checkpoint", "best_model.pth", "--data-dir", DATA_DIR,
                "--legacy-buggy-distance", "--limit", BASELINE_LIMIT])
else:
    print("skipped (RUN_BASELINE_EVAL = False). Verified earlier: 3 cases -> Dice 0.6757 with ground-truth priors.")

## 3. Atlas selection

The atlases are **temporary**: 10 labelled cases standing in for the doctor-segmented atlases that are still to
come. Replacing them means replacing `atlas/atlas_selection.json` and rebuilding the caches; no code changes.

Selection rules (`scripts/select_atlas_cases.py`): drawn from the training split only; all 12 organs present (in
this merged dataset every training case qualifies); round-robin across the three sources so no single scanner or annotation
protocol dominates; deterministic, no random draw.

In [ ]:
if RUN_ATLAS_SELECTION:
    run_script(["scripts/select_atlas_cases.py", "--data-dir", DATA_DIR])

with open("atlas/atlas_selection.json", encoding="utf-8") as f:
    atlas_cases = json.load(f)["cases"]

val_names = {case_name(c["image"]) for c in val_cases}
train_target_names = {case_name(c["image"]) for c in train_targets}
names = [case_name(c["image"]) for c in atlas_cases]
assert not (set(names) & val_names), "an atlas is in the validation split"
assert not (set(names) & train_target_names), "an atlas is still a training target"

rows = []
for i, c in enumerate(atlas_cases):
    reader = sitk.ImageFileReader()
    reader.SetFileName(c["image"])
    reader.ReadImageInformation()
    rows.append({"atlas": i, "case": case_name(c["image"]), "source": source_of(c),
                 "size (x,y,z)": reader.GetSize(), "spacing mm": tuple(round(s, 2) for s in reader.GetSpacing())})
pd.DataFrame(rows)

The atlases differ a lot in resolution (sub-millimetre in-plane with 5 mm slices, versus 1.5 mm isotropic), which
is why later thresholds are in physical units (mm, mm³) rather than voxels.

## 4. Landmarks

Registration here is landmark based. For each organ of each atlas:
- large organs: points on the surface, chosen by farthest-point sampling so they spread evenly;
- small organs (under 8 mL): a few **interior** points instead. Surfaces are where annotators disagree most, and a
  single centroid gave the reliability score nothing to average over: in an early version organ 11 scored 0 %
  reliable and organ 12 100 % in the same atlas pairs, an all-or-nothing outcome from one point each.

All coordinates are in SimpleITK physical (LPS, mm) space, so atlases with different grids and orientations are
directly comparable. The organ names below are inferred from volume and position (AMOS-style order) and have not
been confirmed against the dataset documentation.

In [ ]:
from atlas.reliable_features import extract_atlas_landmarks_lps

ORGAN_NAMES = {1: "spleen?", 2: "R kidney?", 3: "L kidney?", 4: "gallbladder?", 5: "esophagus?", 6: "liver?",
               7: "stomach?", 8: "aorta?", 9: "IVC?", 10: "pancreas?", 11: "R adrenal?", 12: "L adrenal?"}

t0 = time.time()
lm0 = extract_atlas_landmarks_lps(atlas_cases[0]["label"], ORGAN_IDS)
print(f"landmarks for atlas 0 ({names[0]}) in {time.time() - t0:.1f}s")
print(f"{'organ':>5} {'name (inferred)':<14} {'mode':<9} {'points':>6} {'volume mL':>10}")
for oid, info in sorted(lm0.items()):
    print(f"{oid:>5} {ORGAN_NAMES[oid]:<14} {info['mode']:<9} {len(info['points']):>6} {info['volume_mm3'] / 1000:>10.1f}")

fig = plt.figure(figsize=(13, 5))
ax3 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2)
cmap = plt.get_cmap("tab20")
for k, (oid, info) in enumerate(sorted(lm0.items())):
    p = info["points"]
    size = 8 if info["mode"] == "surface" else 45
    ax3.scatter(p[:, 0], p[:, 1], p[:, 2], s=size, color=cmap(k))
    ax2.scatter(p[:, 0], p[:, 2], s=size, color=cmap(k), label=str(oid))
ax3.set_xlabel("x (mm)"); ax3.set_ylabel("y (mm)"); ax3.set_zlabel("z (mm)")
ax2.set_xlabel("x (mm)"); ax2.set_ylabel("z (mm)"); ax2.set_aspect("equal")
ax2.set_title("coronal projection"); ax2.legend(ncol=2, fontsize=7, title="organ")
plt.tight_layout(); plt.show()

## 5. Golden transformation and reliable features

Between two fully labelled atlases, correspondence is easy: organ id is an unambiguous key. The golden
transformation (`atlas/golden_transform.py`) does a rigid Kabsch pre-alignment on organ centroids, matches points
per organ by nearest neighbour, and fits a regularised 3D thin-plate spline (`atlas/tps.py`, written from scratch
because SciPy and SimpleITK have none; includes a Jacobian-determinant folding check).

A landmark's residual r under that spline gives a reliability s = max(T − r, 0) / T with T = 15 mm. Averaged over
all 9 other atlases, this is the landmark's score; only landmarks that agree with most other atlases are kept for
matching in unlabelled targets.

The spline's smoothing term is `K − λI` (for the kernel φ(r) = r the matrix K is conditionally negative definite).
An earlier version used `K + λI`, which made every fit erratic; the cell below showed it as `folding: True`
(details in `experiments/registration_log.md`).

In [ ]:
from atlas.golden_transform import build_golden_transformation

lm3 = extract_atlas_landmarks_lps(atlas_cases[3]["label"], ORGAN_IDS)
golden = build_golden_transformation(lm0, lm3, tolerance_mm=15.0, tps_regularization=5.0)   # maps atlas 3 -> atlas 0
r = golden["residuals"]
print(f"atlas {names[3]} -> {names[0]}: {len(r)} correspondences, residual mean {r.mean():.1f} mm, "
      f"median {np.median(r):.1f} mm, reliable (s>0): {golden['reliable_mask'].mean():.0%}")

pts = golden["source_points"][::7]
detj = golden["transform"].jacobian_determinant(pts)
print("Jacobian determinant on a sample of control points:", f"[{detj.min():.2f}, {detj.max():.2f}]",
      "| folding:", bool((detj <= 0).any()))

rows = []
for oid in ORGAN_IDS:
    m = golden["organ_ids"] == oid
    rows.append({"organ": oid, "points": int(m.sum()), "mean residual mm": round(float(r[m].mean()), 1),
                 "reliable": f"{golden['reliable_mask'][m].mean():.0%}"})
pd.DataFrame(rows).set_index("organ").T

In [ ]:
from atlas.reliable_features import load_atlas_library

LIB_PATH = REPO_ROOT / "cache" / "atlas_library" / "atlas_library.npz"
if RUN_LIBRARY_BUILD or not LIB_PATH.exists():
    run_script(["scripts/build_atlas_library.py"])
library = load_atlas_library(str(LIB_PATH))

all_scores = np.concatenate([e["all_scores"] for e in library])
print(f"{len(all_scores)} landmarks over {len(library)} atlases; median score {np.median(all_scores):.2f}, "
      f"p90 {np.percentile(all_scores, 90):.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].hist(all_scores, bins=40, color="#4c78a8")
ax[0].axvline(0.3, color="crimson", ls="--", label="keep threshold 0.3")
ax[0].set_xlabel("reliability score"); ax[0].set_ylabel("landmarks"); ax[0].legend()
kept = {oid: np.mean([(e["organ_ids"] == oid).sum() for e in library]) for oid in ORGAN_IDS}
ax[1].bar([str(o) for o in kept], list(kept.values()), color="#59a14f")
ax[1].set_xlabel("organ"); ax[1].set_ylabel("kept landmarks per atlas")
plt.tight_layout(); plt.show()

print("selected / total per atlas:", [f"{len(e['points'])}/{len(e['all_points'])}" for e in library])

With the corrected TPS the score median is 0.43 and the threshold of 0.3 keeps about three quarters of the landmarks
(~300 per atlas), so it is now barely selective. It was picked when the median was 0.32, and is an open tuning item
(to be tuned on training targets, not validation). Organs 11 and 12 keep only ~4 to 5 landmarks each, so the small
organs are the thinnest part of the registration.

## 6. Atlas → target registration and label warping

For an unlabelled target CT, each atlas is registered by (`atlas/registration.py`):

1. **Coarse alignment** on body masks: a z-offset search (scans cover different stretches of the body), then a
   similarity fit, then an affine accepted only if every axis scale stays in 0.8 to 1.25. An earlier unconstrained
   affine collapsed one atlas to 0.19× height when the fields of view differed.
2. **Patch matching**: each reliable atlas landmark is located in the target by normalised cross-correlation of a
   9³-voxel soft-tissue patch in a local window (a batched GPU convolution, `atlas/matching.py`), in two passes with a
   tighter second window.
3. **RANSAC affine** over those matches, rejecting mirror images and implausible scales (`atlas/ransac.py`).
4. **Smoothing TPS** on the inliers, fitted target → atlas so labels can be pulled back without inverting a spline,
   with a folding check and an explicit, recorded fallback (`tps`, `ransac_affine_only`, `coarse_affine_only`). The
   smoothing weight λ starts at 5 and is multiplied by 4 while the field folds; so far every registration has stayed at 5.
5. **Label warping**: the atlas label is resampled into the target's 2 mm grid by nearest neighbour; the 10 warped
   labels are cached as one 4D file per target (`atlas/label_warping.py`).

The target's own label is never used. With `--eval-gt` it is read afterwards, only to report how good the
atlas-only vote is.

In [ ]:
if RUN_REGISTRATION:
    run_script(["scripts/register_targets.py", "--split", "val", "--limit", N_REGISTER, "--eval-gt", "--qa-png"])

reg_dir = REPO_ROOT / "cache" / "registrations" / "tps"
records = sorted(reg_dir.glob("*.json")) if reg_dir.exists() else []
rows = []
for p in records:
    rec = json.loads(p.read_text(encoding="utf-8"))
    ev = rec.get("evaluation_only")
    row = {"case": p.stem, "split": "val" if p.stem in val_names else "train", **rec["status_counts"],
           "time s": rec["elapsed_s"]}
    if ev:
        row["atlas-only Dice"] = round(ev["majority_vote_mean_dice"], 3)
        row.update({f"o{k}": round(v, 2) for k, v in ev["majority_vote_dice"].items()})
    rows.append(row)
registration_table = pd.DataFrame(rows)
print(f"{len(records)} cached registrations in {reg_dir}")
registration_table

In [ ]:
qa_dir = REPO_ROOT / "cache" / "qa" / "tps"
pngs = sorted(qa_dir.glob("*.png"))[:2] if qa_dir.exists() else []
for p in pngs:
    plt.figure(figsize=(16, 6.5))
    plt.imshow(plt.imread(p)); plt.axis("off"); plt.title(p.stem)
    plt.tight_layout(); plt.show()
if not pngs:
    print("no QA images yet: set RUN_REGISTRATION = True (it writes them)")

### One atlas, one target, every stage

Runs a single registration in-process so the intermediate quantities are visible: what the coarse stage chose,
how many landmark matches survive thresholding and RANSAC in each pass, and whether the TPS needed extra smoothing
to avoid folding.

In [ ]:
from atlas.geometry import read_ct, resample_isotropic
from atlas.registration import RegistrationConfig, prepare_target, register_atlas_to_target

reg = None
if RUN_SINGLE_PAIR:
    cfg = RegistrationConfig()
    target_case = val_cases[0]
    t0 = time.time()
    prep = prepare_target(read_ct(target_case["image"]), cfg)
    atlas_ct = resample_isotropic(read_ct(library[0]["image"]), cfg.match_spacing_mm, sitk.sitkLinear, -1000.0)
    reg = register_atlas_to_target(atlas_ct, library[0]["points"], prep, cfg)
    q = reg.qa
    print(f"target {case_name(target_case['image'])}  <-  atlas {names[0]}   ({time.time() - t0:.0f}s)")
    c = q["coarse"]
    print(f"coarse    : stage={c['stage']}, z offset {c['z_offset_mm']:.0f} mm (overlap {c['z_overlap']:.0%}), "
          f"axis scales {[round(s, 2) for s in c['affine_scales']]}")
    for key in ("pass1", "pass2"):
        if key in q:
            s = q[key]
            print(f"{key}     : window +-{s['search_radius']} vox, {s['num_candidates']} candidates, "
                  f"{s['ransac'].get('num_inliers', 0)} RANSAC inliers, median NCC {s['median_ncc_candidates']:.2f}")
    print(f"used pass {q['pass_used']} | status: {reg.status} | TPS lambda {q.get('tps_regularization_used')} "
          f"| fold fraction {q.get('fold_fraction')}")

    d = reg.debug
    plt.figure(figsize=(6, 3.2))
    plt.hist(d["ncc"], bins=40, alpha=0.6, label="all landmarks")
    plt.hist(d["ncc"][d["inliers"]], bins=40, alpha=0.8, label="RANSAC inliers")
    plt.axvline(cfg.min_ncc, color="k", ls="--", label="NCC threshold")
    plt.xlabel("peak NCC of best patch match"); plt.ylabel("landmarks"); plt.legend()
    plt.tight_layout(); plt.show()
else:
    print("skipped (RUN_SINGLE_PAIR = False)")

### What the registration currently achieves

Atlas-only majority vote on the first two validation targets (evaluation only, target labels never used to build
the priors); details and open tuning items in `experiments/registration_log.md`.

| Variant | amos_0278 | s0703 |
|---|---|---|
| first pipeline (unconstrained coarse affine) * | 0.334 | 0.328 |
| constrained coarse stage, second matching pass * | 0.321 | 0.321 |
| + B-spline refinement on whole-body mutual information * | 0.265 | 0.316 |
| **after fixing the TPS smoothing sign (current)** | **0.445** | **0.481** |
| current + B-spline refinement | 0.443 | 0.458 |

\* produced with the faulty TPS, kept only for history.

What this says:
- The TPS sign error had hidden the TPS's benefit: fixing it raised atlas-only Dice from about 0.32 to 0.45 to 0.48
  with no other change. For a single atlas on amos_0278 the stages now order coarse affine < RANSAC affine < TPS
  (for example 0.36 → 0.44 → 0.54).
- 49 of 50 cached registrations end in a TPS (one falls back to the coarse affine), all at the first smoothing value
  and without folding.
- Dense B-spline refinement does not help: whole-body mutual information improves in every pair while organ overlap
  does not, so it is not used (it stays available as `--refinement bspline`).
- Still weak: organ 4 is 0.00 on both cases and organs 11 and 12 are near 0 on one of them. Two cases say little
  about individual organs.
- Open, all to be tuned on non-atlas *training* targets: golden-transform λ and tolerance, reliability threshold,
  registration TPS λ, RANSAC threshold and patch size.

## 7. Priors: P(v), D(v), w(v)

`AtlasPriorsd` (in `data/transforms.py`) turns the cached warped-atlas stack into the model's inputs **after** all
cropping, resizing and augmentation, so the priors stay aligned with the CT the model actually sees:

- P_c(v): fraction of the 10 atlases that label voxel v as organ c,
- w(v) = 1 − H(v)/log C: confidence from the entropy of that vote,
- D_c(v): distance in mm to the atlas-derived region of organ c, clipped at 50 mm and divided by 50.

The organ region is where P_c is at least half its maximum. That rule keeps D defined for small organs whose atlases
rarely agree (a fixed 0.5 cut would leave them empty). The warped stack is loaded by MONAI onto the same 96³ grid as
the image through its tracked affine (`ResampleToMatchd`), then flipped and rotated together with it.

In [ ]:
from data.splits import attach_atlas_priors

cached_val, missing_val = attach_atlas_priors(val_cases, "tps")
print(f"{len(cached_val)} of {len(val_cases)} validation cases have cached atlas stacks")

tf_atlas = build_transforms(IMG_SIZE, NUM_CLASSES, train=False, prior_source="atlas")
demo_case = demo = None
if cached_val:
    demo_case = cached_val[0]
    demo = tf_atlas(demo_case)
    P, D, W = demo["prob"], demo["dist"], demo["conf"]
    print(f"case {case_name(demo_case['image'])}: image {tuple(demo['image'].shape)}, P {tuple(P.shape)}, "
          f"D {tuple(D.shape)}, w {tuple(W.shape)}")
    print(f"P sums to {float(P.sum(0).mean()):.3f} | D in [{float(D.min()):.2f}, {float(D.max()):.2f}] "
          f"| w in [{float(W.min()):.2f}, {float(W.max()):.2f}], mean {float(W.mean()):.2f}")
    print("keys the model sees:", [k for k in demo if k in ("image", "prob", "dist", "conf")],
          "| label only as loss target:", "label" in demo)
else:
    print("no cached stacks: set RUN_REGISTRATION = True first")

In [ ]:
if demo is not None:
    img = demo["image"][0].numpy()
    lab = demo["label"].argmax(0).numpy()
    voted = demo["prob"].argmax(0).numpy()
    conf = demo["conf"][0].numpy()
    dist6 = demo["dist"][5].numpy()                    # channel 5 = organ 6 (distance to its atlas region)

    organ_voxels = np.argwhere(voted > 0)              # slice through where the PRIOR puts organs, not the label
    cx, cy, cz = organ_voxels.mean(axis=0).round().astype(int)
    views = {"axial": lambda a: a[:, :, cz].T, "coronal": lambda a: a[:, cy, :].T}

    fig, axes = plt.subplots(2, 5, figsize=(17, 7))
    for r, (vname, sl) in enumerate(views.items()):
        panels = [("CT", sl(img), "gray", None), ("supervised target (label)", sl(lab), "nipy_spectral", (0, 12)),
                  ("atlas vote P argmax", sl(voted), "nipy_spectral", (0, 12)),
                  ("confidence w(v)", sl(conf), "magma", (0, 1)), ("D for organ 6", sl(dist6), "viridis", (0, 1))]
        for c, (title, arr, cmapname, lim) in enumerate(panels):
            a = axes[r, c]
            a.imshow(arr, cmap=cmapname, origin="lower", **({"vmin": lim[0], "vmax": lim[1]} if lim else {}))
            a.set_title(f"{vname}: {title}", fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()

    dice = []
    for k in ORGAN_IDS:
        p, g = voted == k, lab == k
        dice.append(2 * (p & g).sum() / max(p.sum() + g.sum(), 1))
    plt.figure(figsize=(7, 3))
    plt.bar([str(k) for k in ORGAN_IDS], dice, color="#e15759")
    plt.ylim(0, 1); plt.xlabel("organ"); plt.ylabel("Dice")
    plt.title(f"atlas-only vote vs label in the 96^3 training space (evaluation only), mean {np.mean(dice):.2f}")
    plt.tight_layout(); plt.show()
    print(f"mean atlas-only Dice in the 96^3 training space: {np.mean(dice):.3f}  (2 mm registration grid, same case: "
          f"{registration_table.set_index('case').loc[case_name(demo_case['image']), 'atlas-only Dice']})")

The mean here is close to, but a little below, the value on the 2 mm registration grid (0.42 versus 0.445 for this
case in the last run). That shows warping, cropping, resizing and augmentation keep the prior aligned with the CT; the
coarser 96³ grid probably costs small organs a little, which was not investigated. The prior is informative for large organs but far from the label
for small ones; how much the network can do with it is what Experiment 1 tests.

## 8. Model and Experiment 1

The architecture is unchanged from the prototype: `AtlasGuidedViTUNETR` is a MONAI UNETR whose 12 transformer
blocks use cross-attention, with image tokens as queries and **anatomical tokens** as keys and values. An MLP
builds one anatomical token per 16³ patch from the pooled [P, D] values. Loss (`DiceCELoss`), optimiser, learning
rate and augmentation are also the prototype's, so a change in Dice can be attributed to the prior alone.

In [ ]:
from models.segmentation_model import AtlasGuidedViTUNETR

model = None
if Path("best_model.pth").exists():
    model = AtlasGuidedViTUNETR(in_channels=1, out_channels=NUM_CLASSES, img_size=IMG_SIZE, feature_size=16,
                                hidden_size=768, mlp_dim=3072, num_heads=12, proj_type="perceptron",
                                norm_name="instance", res_block=True).to(device)
    ckpt = torch.load("best_model.pth", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt)
    print(f"epoch-101 weights loaded, {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")
else:
    print("best_model.pth not found (it is not in git): the epoch-101 comparisons in this section are skipped")

### Same weights, same case, two kinds of prior

The epoch-101 network evaluated on one validation case, once with its ground-truth prior (what it was trained on)
and once with the atlas prior (what it would get on a new scan). Note this is a single case.

In [ ]:
from monai.data import DataLoader, Dataset
from training.engine import validate, train_one_epoch

def one_case_dice(sample, keys):
    mean, per_organ = validate(model, DataLoader(Dataset([sample]), batch_size=1), device, keys, amp=False)
    return mean, per_organ

if demo is not None and model is not None:
    plain_case = {k: demo_case[k] for k in ("image", "label")}
    sample_gt = tf_gt(plain_case)
    d_gt, per_gt = one_case_dice(sample_gt, PRIOR_KEYS["gt"])
    d_atlas, per_atlas = one_case_dice(demo, PRIOR_KEYS["atlas"])
    print(f"case {case_name(demo_case['image'])} | epoch-101 weights")
    print(f"  ground-truth prior (prototype) : mean Dice {d_gt:.3f}")
    print(f"  atlas prior                    : mean Dice {d_atlas:.3f}")
    pd.DataFrame({"GT prior": per_gt, "atlas prior": per_atlas}, index=[f"o{k}" for k in ORGAN_IDS]).round(2).T
else:
    print("needs a cached registration (see section 6) and best_model.pth")

In the local smoke test the epoch-101 weights scored **0.12 with atlas priors on two validation cases, versus 0.68
with ground-truth priors on the first three (0.65 on the full set)**. Different case counts, so only the gap is
meaningful. The prototype mostly learned to read the prior, so **0.6505 cannot be compared with any atlas-prior
result**. That 0.12 did not move when the prior itself improved (0.120 before the TPS fix, 0.119 after, while atlas-only
Dice rose from about 0.32 to about 0.46); an untested explanation is that the model sees priors only through 16³-patch
pooling and was tuned to exact binary priors. Experiment 1 fine-tunes from these weights to see how far the network
can adapt; a model trained with no prior at all (stock UNETR) is still needed as the honest reference.
(`Atlas_Comparison_Models.ipynb` defines stock UNETR and UNet runs but only with a 5-epoch setup, so that
reference does not exist yet.)

### A few training steps, in-process (demonstration only)

Mirrors the first steps of `scripts/train.py` on cached training targets, on this 4 GB GPU with fp16 autocast.
Nothing is saved, and it **modifies the `model` object in this notebook**, so re-run the model cell to get the
epoch-101 weights back.

In [ ]:
from monai.losses import DiceCELoss

if RUN_TRAIN_SMOKE and model is not None:
    train_cached, _ = attach_atlas_priors(train_targets, "tps")
    val_subset = cached_val[:2]
    train_subset = train_cached[:3]
    print(f"{len(train_subset)} training targets and {len(val_subset)} validation cases with cached priors")

    val_loader = DataLoader(Dataset(val_subset, tf_atlas), batch_size=1)
    train_loader = DataLoader(Dataset(train_subset, build_transforms(IMG_SIZE, NUM_CLASSES, train=True,
                                                                     prior_source="atlas")),
                              batch_size=1, shuffle=True)
    criterion = DiceCELoss(to_onehot_y=False, softmax=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    keys = PRIOR_KEYS["atlas"]

    before, _ = validate(model, val_loader, device, keys, amp=use_amp)
    loss = train_one_epoch(model, train_loader, optimizer, criterion, device, keys, scaler=scaler, amp=use_amp,
                           max_steps=3, log_every=1)
    after, per = validate(model, val_loader, device, keys, amp=use_amp)
    print(f"validation Dice with atlas priors: {before:.3f} before  ->  {after:.3f} after 3 steps (train loss {loss:.3f})")
    print("per organ after:", {f"o{k}": None if s is None else round(s, 2) for k, s in zip(ORGAN_IDS, per)})
    torch.cuda.empty_cache()
else:
    print("skipped (RUN_TRAIN_SMOKE = False). Local smoke test earlier: 0.119 -> 0.198 after 3 steps on 3 targets "
          "(2 validation cases), which shows the loop works, not that it converges.")

### Tracked runs

`scripts/train.py` is what real runs use. Each run gets its own folder `experiments/<experiment>/<run_name>/` with
`config.json`, `command.txt`, `git.txt` and diff, `metrics.csv`, `summary.json`, `training.log` and checkpoints, and an
existing run folder is never overwritten. Epoch 0 in `metrics.csv` is the epoch-101 weights scored with atlas priors
before any fine-tuning, the reference point for how much the prototype relied on ground-truth priors. The script
refuses to start if some cases have no registration cache (`--allow-missing` is for smoke tests).

Full Experiment 1 needs every target registered first (about 670 targets at ~2 min each, sharded across processes
with `--num-shards`/`--shard-index`); the commands are in `RUN_GUIDE.md`. A run costs hours, so use a server.

In [ ]:
if RUN_TRAIN_SCRIPT:
    run_script(["scripts/train.py", *TRAIN_SCRIPT_ARGS])

frames = {}
for p in sorted((REPO_ROOT / "experiments").glob("*/*/metrics.csv")):
    df = pd.read_csv(p)
    if {"epoch", "train_loss", "mean_dice"} <= set(df.columns):   # scripts/train.py runs; section 10 shows the v2 runs
        frames[f"{p.parent.parent.name}/{p.parent.name}"] = df
print("runs found:", list(frames) or "none")
for name, df in frames.items():
    print("\n" + name)
    print(df[["epoch", "train_loss", "mean_dice"]].round(4).to_string(index=False))

<!-- v2-model-section -->
## 9. Redesigned model: atlas-aware ViT-UNETR (prototype of `new_model_plan.md`)

Section 8 showed that the epoch-101 network ignores (or is misled by) a realistic atlas prior. This section builds the
planned model **here in the notebook only**; nothing in `models/` changes until it passes. Every change is a flag and
is a no-op at initialisation, so the redesigned model starts from exactly the epoch-101 function:

| Plan item | What is built here | Flag | Init value |
|---|---|---|---|
| A | train and validate on atlas priors only | data side (`prior_source="atlas"`) | – |
| B | `PriorDropoutd`: with p = 0.3 per training sample, P → uniform, D → 1, w → 0 | transform | – |
| C | pooled w(v) as a 26th token-MLP input + a learnable gate g_i on each block's **cross-attention residual** only | `conf_gating` | new MLP column 0, g_i = 1 |
| D | `encoder1` (`conv1` and residual `conv3`) also reads [P, D, w] at voxel level: 1 → 27 input channels | `voxel_prior` | new channels 0 |
| E | `logits += γ_c · w · log(P + 1e-3)` | `output_fusion` | γ = 0 |
| F | `DiceCE + λ · mean(w · KL(P ‖ softmax(logits)))`, λ ramped over 10 epochs | loss | λ = 0 at epoch 0 |

**One correction to the plan.** It says to load epoch-101 with `strict=False`. That fails: C and D change the *shape*
of two existing tensors (`anatomical_prior_generator.mlp.0.weight` 25 → 26 columns, `encoder1.layer.conv1/conv3`
1 → 27 input channels), and PyTorch rejects shape mismatches even when `strict=False`. `load_baseline_weights` below
copies the old weights into the leading slice, zeroes the new slice, and keeps the identity init of parameters that
did not exist (gates, γ). It then loads with `strict=True`, so nothing is silently skipped.

Two smaller deviations, both deliberate: the KL is **summed** over classes per voxel (the plan's `.mean()` over all
elements divides it by 13), and it is computed with `log_softmax` in float32 rather than `log(softmax + 1e-7)` under
fp16 autocast. If no confidence map is passed (ground-truth priors), w = 1, which is exactly the entropy confidence
of a one-hot prior.

In [ ]:
#v2-cell definitions
import torch.nn as nn
import torch.nn.functional as F
import monai.transforms as mt
from monai.networks.blocks import TransformerBlock, UnetrBasicBlock
from models.segmentation_model import AtlasGuidedViTUNETR, AnatomicalPriorGenerator
from data.splits import attach_atlas_priors
from data.transforms import build_transforms

LOG_PRIOR_EPS = 1e-3


class PriorDropoutd(mt.RandomizableTransform, mt.MapTransform):
    # (B) Training only, after AtlasPriorsd: with probability `prob` the sample gets "no atlas information".
    def __init__(self, prob=0.3, keys=("prob", "dist", "conf")):
        mt.RandomizableTransform.__init__(self, prob=prob)
        mt.MapTransform.__init__(self, keys)

    def __call__(self, data):
        d = dict(data)
        self.randomize(None)
        if self._do_transform:
            p, dist, conf = self.keys
            d[p] = torch.full_like(d[p], 1.0 / d[p].shape[0])
            d[dist] = torch.ones_like(d[dist])
            d[conf] = torch.zeros_like(d[conf])
        return d


class GatedTransformerBlock(TransformerBlock):
    # (C) Stock MONAI block plus one scalar that scales only the cross-attention residual (1 = stock block).
    def __init__(self, *args, **kwargs):
        super().__init__(*args, with_cross_attention=True, **kwargs)
        self.cross_gate = nn.Parameter(torch.ones(()))

    def forward(self, x, context=None, attn_mask=None):
        x = x + self.attn(self.norm1(x), attn_mask=attn_mask)
        x = x + self.cross_gate * self.cross_attn(self.norm_cross_attn(x), context=context)
        return x + self.mlp(self.norm2(x))


class ConfidencePriorGenerator(AnatomicalPriorGenerator):
    # (C) Same pooling and MLP; pooled w is appended after D, so MLP inputs are [P(13), D(12), w(1)].
    def __init__(self, img_size, patch_size, num_prob_channels, num_dist_channels, hidden_size):
        super().__init__(img_size, patch_size, num_prob_channels, num_dist_channels + 1, hidden_size)

    def forward(self, probability_maps, distance_maps, confidence_maps):
        return super().forward(probability_maps, torch.cat([distance_maps, confidence_maps], dim=1))


class AtlasAwareViTUNETR(AtlasGuidedViTUNETR):
    def __init__(self, in_channels, out_channels, img_size, feature_size=16, hidden_size=768, mlp_dim=3072,
                 num_heads=12, proj_type="perceptron", norm_name="instance", res_block=True, dropout_rate=0.0,
                 qkv_bias=False, conf_gating=False, voxel_prior=False, output_fusion=False):
        super().__init__(in_channels, out_channels, img_size, feature_size=feature_size, hidden_size=hidden_size,
                         mlp_dim=mlp_dim, num_heads=num_heads, proj_type=proj_type, norm_name=norm_name,
                         res_block=res_block, dropout_rate=dropout_rate, qkv_bias=qkv_bias)
        self.conf_gating, self.voxel_prior, self.output_fusion = conf_gating, voxel_prior, output_fusion
        n_prob, n_dist = out_channels, out_channels - 1
        if conf_gating:
            self.anatomical_prior_generator = ConfidencePriorGenerator(img_size, self.patch_size, n_prob, n_dist,
                                                                       hidden_size)
            self.vit.blocks = nn.ModuleList(
                [GatedTransformerBlock(hidden_size, mlp_dim, num_heads, dropout_rate, qkv_bias=qkv_bias)
                 for _ in range(len(self.vit.blocks))])
        if voxel_prior:
            self.encoder1 = UnetrBasicBlock(spatial_dims=3, in_channels=in_channels + n_prob + n_dist + 1,
                                            out_channels=feature_size, kernel_size=3, stride=1,
                                            norm_name=norm_name, res_block=res_block)
        if output_fusion:
            self.output_fusion_weight = nn.Parameter(torch.zeros(out_channels))

    def forward(self, x_in, probability_maps, distance_maps, confidence_maps=None):
        if confidence_maps is None:
            confidence_maps = torch.ones_like(probability_maps[:, :1])
        if self.conf_gating:
            tokens = self.anatomical_prior_generator(probability_maps, distance_maps, confidence_maps)
        else:
            tokens = self.anatomical_prior_generator(probability_maps, distance_maps)
        x, hidden_states_out = self.vit(x_in, tokens)

        enc_in = torch.cat([x_in, probability_maps, distance_maps, confidence_maps], 1) if self.voxel_prior else x_in
        enc1 = self.encoder1(enc_in)
        enc2 = self.encoder2(self.proj_feat(hidden_states_out[3]))
        enc3 = self.encoder3(self.proj_feat(hidden_states_out[6]))
        enc4 = self.encoder4(self.proj_feat(hidden_states_out[9]))
        dec3 = self.decoder5(self.proj_feat(x), enc4)
        dec2 = self.decoder4(dec3, enc3)
        dec1 = self.decoder3(dec2, enc2)
        logits = self.out(self.decoder2(dec1, enc1))

        if self.output_fusion:
            log_prior = torch.log(probability_maps.float() + LOG_PRIOR_EPS)
            gamma = self.output_fusion_weight.view(1, -1, 1, 1, 1)
            logits = logits + gamma * confidence_maps.float() * log_prior
        return logits

    def new_parameters(self):
        out = {}
        if self.conf_gating:
            out["cross_gates"] = torch.stack([b.cross_gate.detach() for b in self.vit.blocks])
        if self.output_fusion:
            out["gamma"] = self.output_fusion_weight.detach()
        return out


def load_baseline_weights(model, path="best_model.pth"):
    # Old weights go into the leading input slice of widened tensors, the new slice is zeroed;
    # tensors absent from the checkpoint keep their identity init. Returns (widened keys, new keys).
    sd = torch.load(path, map_location="cpu", weights_only=False)
    sd = sd.get("model_state_dict", sd)
    own = model.state_dict()
    unexpected = [k for k in sd if k not in own]
    if unexpected:
        raise KeyError(f"checkpoint keys not in model: {unexpected[:5]}")
    merged, widened = {}, []
    for k, cur in own.items():
        if k not in sd:
            merged[k] = cur
            continue
        old = sd[k].to(cur.dtype)
        if old.shape != cur.shape:
            if old.ndim != cur.ndim or old.shape[0] != cur.shape[0] or old.shape[2:] != cur.shape[2:] \
                    or old.shape[1] > cur.shape[1]:
                raise ValueError(f"{k}: cannot widen {tuple(old.shape)} -> {tuple(cur.shape)}")
            w = torch.zeros_like(cur)
            w[:, :old.shape[1]] = old
            merged[k] = w
            widened.append(k)
        else:
            merged[k] = old
    model.load_state_dict(merged, strict=True)
    return widened, [k for k in own if k not in sd]


def atlas_consistency_loss(logits, prob, conf):
    # (F) mean over voxels of w(v) * KL(P(v) || softmax(logits)(v)), KL summed over classes, float32.
    log_q = F.log_softmax(logits.float(), dim=1)
    kl = F.kl_div(log_q, prob.float(), reduction="none").sum(dim=1, keepdim=True)
    return (conf.float() * kl).mean()


def consistency_lambda(epoch, weight, ramp_epochs=10):
    return weight if ramp_epochs <= 0 else weight * min(1.0, epoch / ramp_epochs)


def build_v2_transforms(train, prior_dropout=0.3, **kw):
    tf = build_transforms(IMG_SIZE, NUM_CLASSES, train=train, prior_source="atlas", **kw)
    return mt.Compose([tf, PriorDropoutd(prior_dropout)]) if train and prior_dropout > 0 else tf


def v2_train_steps(model, loader, optimizer, criterion, device, scaler=None, amp=False, max_steps=None,
                   consistency_weight=0.0, epoch=0, ramp_epochs=10, log=print):
    model.train()
    lam = consistency_lambda(epoch, consistency_weight, ramp_epochs)
    total, steps = 0.0, 0
    for batch in loader:
        img, lab = batch["image"].to(device), batch["label"].to(device)
        prob, dist, conf = batch["prob"].to(device), batch["dist"].to(device), batch["conf"].to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            logits = model(img, prob, dist, conf)
        sup = criterion(logits.float(), lab)
        cons = atlas_consistency_loss(logits, prob, conf) if lam > 0 else torch.zeros((), device=device)
        loss = sup + lam * cons
        skipped = ""
        if scaler is not None and amp:
            scale = scaler.get_scale()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            if scaler.get_scale() < scale:
                skipped = f"  [skipped by GradScaler: inf/NaN grads at scale {scale:.0f}]"
        else:
            loss.backward(); optimizer.step()
        total += loss.item(); steps += 1
        log(f"    step {steps}: DiceCE {sup.item():.4f} + {lam:.3f} x KL {cons.item():.4f}{skipped}")
        if max_steps is not None and steps >= max_steps:
            break
    return total / max(steps, 1)


@torch.no_grad()
def v2_validate(model, loader, device, amp=False, use_conf=True):
    from monai.metrics import DiceMetric
    model.eval()
    metric = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=True)
    for batch in loader:
        args = [batch[k].to(device) for k in ("image", "prob", "dist")]
        if use_conf:
            args.append(batch["conf"].to(device))
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"):
            logits = model(*args)
        pred = F.one_hot(logits.float().argmax(1), NUM_CLASSES).permute(0, 4, 1, 2, 3).float()
        metric(y_pred=pred, y=batch["label"].to(device))
    per_class, not_nans = metric.aggregate()
    per_organ = [float(s) if n else None for s, n in zip(per_class.tolist(), not_nans.tolist())]
    return per_class[not_nans.bool()].mean().item(), per_organ


MODEL_KW = dict(in_channels=1, out_channels=NUM_CLASSES, img_size=IMG_SIZE, feature_size=16, hidden_size=768,
                mlp_dim=3072, num_heads=12, proj_type="perceptron", norm_name="instance", res_block=True)
ALL_FLAGS = dict(conf_gating=True, voxel_prior=True, output_fusion=True)

n_old = sum(p.numel() for p in AtlasGuidedViTUNETR(**MODEL_KW).parameters())
n_new = sum(p.numel() for p in AtlasAwareViTUNETR(**MODEL_KW, **ALL_FLAGS).parameters())
print(f"parameters: baseline {n_old / 1e6:.2f} M, redesigned {n_new / 1e6:.2f} M (+{n_new - n_old:,})")

<!-- v2-model-section -->
### Tests

Run on one cached validation case with atlas priors (random tensors if no case is cached). Each check is an assertion;
the table at the end summarises them.

1. **Flags off** ⇒ the redesigned class has exactly the baseline's state-dict keys and shapes, loads `best_model.pth` (or the random fallback)
   with `strict=True`, and gives the baseline's output.
2. **Each flag alone, and all three** ⇒ after `load_baseline_weights`, the output equals the epoch-101 output
   (|Δlogit| ≤ 1e-5 × logit scale, argmax identical everywhere). Run in exact fp32: with cuDNN's default TF32 the
   widened `encoder1` uses a different kernel and drifts by ~0.07 logits, which is rounding, not a model difference.
3. **Every new parameter gets a non-zero gradient** (gates, γ, new MLP column, new `encoder1` channels). A parameter
   initialised to zero that receives no gradient would stay a no-op forever.
4. **Gate semantics**: with every g_i = 0 (and C only), the output no longer depends on the prior at all.
5. **Fusion formula**: with γ ≠ 0 the logits change by exactly γ_c · w · log(P + ε); with w = 0 (a dropped prior) by 0.
6. **Prior dropout**: p = 1 gives (uniform, 1, 0), p = 0 is the identity, p = 0.3 fires ≈ 30 % of the time.
7. **Consistency loss**: ≥ 0, ≈ 0 when softmax(logits) = P, exactly 0 when w = 0; λ ramps 0 → 0.1 over 10 epochs.

In [ ]:
#v2-cell tests
from monai.data import DataLoader, Dataset
from monai.losses import DiceCELoss

# The equivalence properties hold for ANY baseline weights, so without best_model.pth (it is not in git) the checks
# use a seeded random baseline instead.
BASELINE_CKPT = "best_model.pth"
if not os.path.exists(BASELINE_CKPT):
    torch.manual_seed(0)
    BASELINE_CKPT = str(REPO_ROOT / "cache" / "random_baseline_for_tests.pth")
    os.makedirs(os.path.dirname(BASELINE_CKPT), exist_ok=True)
    torch.save(AtlasGuidedViTUNETR(**MODEL_KW).state_dict(), BASELINE_CKPT)
    print("best_model.pth not found: the checks use a seeded random baseline")

results = []
def check(name, ok, detail=""):
    results.append({"test": name, "result": "PASS" if ok else "FAIL", "detail": detail})
    print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))

cached_val, _ = attach_atlas_priors(val_cases, "tps")
tf_v2_val = build_v2_transforms(train=False)
if cached_val:
    s = tf_v2_val(cached_val[0])
    test_name = case_name(cached_val[0]["image"])
    inputs = [s[k].unsqueeze(0).float() for k in ("image", "prob", "dist", "conf")]
else:
    g = torch.Generator().manual_seed(0)
    p = torch.rand((1, NUM_CLASSES, *IMG_SIZE), generator=g); p = p / p.sum(1, keepdim=True)
    inputs = [torch.rand((1, 1, *IMG_SIZE), generator=g), p, torch.rand((1, NUM_CLASSES - 1, *IMG_SIZE), generator=g),
              torch.rand((1, 1, *IMG_SIZE), generator=g)]
    test_name = "random tensors"
inputs = [t.to(device) for t in inputs]
img_t, prob_t, dist_t, conf_t = inputs
print("test input:", test_name)

# --- 1/2: equivalence with epoch-101 --------------------------------------------------------------
# cuDNN's default TF32 convs round to a 10-bit mantissa and pick a different kernel for 27 input channels than
# for 1, so the widened encoder1 differs by ~1e-3 (0.07 at the logits) although the maths is identical.
# Exact fp32 is used for the equivalence checks only.
tf32_state = (torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32)
torch.backends.cudnn.allow_tf32 = torch.backends.cuda.matmul.allow_tf32 = False
base =AtlasGuidedViTUNETR(**MODEL_KW).to(device).eval()
base.load_state_dict(torch.load(BASELINE_CKPT, map_location=device, weights_only=False), strict=True)
with torch.no_grad():
    ref = base(img_t, prob_t, dist_t).float()
ref_keys = {k: v.shape for k, v in base.state_dict().items()}
del base; torch.cuda.empty_cache()

def compare(out, name):
    diff = (out - ref).abs().max().item()
    agree = (out.argmax(1) == ref.argmax(1)).float().mean().item()
    check(name, diff <= 1e-5 * ref.abs().max().item() and agree == 1.0,
          f"max |Δlogit| {diff:.2e} (logit scale {ref.abs().max().item():.1f}), argmax agreement {agree:.6f}")

m = AtlasAwareViTUNETR(**MODEL_KW).to(device).eval()
same_keys = {k: v.shape for k, v in m.state_dict().items()} == ref_keys
m.load_state_dict(torch.load(BASELINE_CKPT, map_location=device, weights_only=False), strict=True)
with torch.no_grad():
    compare(m(img_t, prob_t, dist_t, conf_t).float(), "1. flags off: same keys/shapes, strict load, same output")
check("1. flags off: state dict identical to baseline", same_keys)
del m; torch.cuda.empty_cache()

for flags in ({"conf_gating": True}, {"voxel_prior": True}, {"output_fusion": True}, ALL_FLAGS):
    label = "+".join(k for k in flags) if flags is not ALL_FLAGS else "all flags"
    m = AtlasAwareViTUNETR(**MODEL_KW, **flags).to(device).eval()
    widened, new = load_baseline_weights(m, BASELINE_CKPT)
    with torch.no_grad():
        compare(m(img_t, prob_t, dist_t, conf_t).float(), f"2. {label}: identical to epoch-101 at init")
    print(f"     widened: {widened or '-'} | new: {len(new)} tensors")
    del m; torch.cuda.empty_cache()
torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32 = tf32_state

# --- 3: gradients reach every new parameter ------------------------------------------------------
m = AtlasAwareViTUNETR(**MODEL_KW, **ALL_FLAGS).to(device).train()
load_baseline_weights(m, BASELINE_CKPT)
with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
    logits = m(img_t, prob_t, dist_t, conf_t)
lab_t = (s["label"].unsqueeze(0).float().to(device) if cached_val
         else F.one_hot(prob_t.argmax(1), NUM_CLASSES).permute(0, 4, 1, 2, 3).float())
loss = DiceCELoss(to_onehot_y=False, softmax=True)(logits.float(), lab_t) \
       + 0.1 * atlas_consistency_loss(logits, prob_t, conf_t)
loss.backward()
grads = {
    "cross gates": torch.stack([b.cross_gate.grad for b in m.vit.blocks]).abs().min().item(),
    "fusion γ (organs)": m.output_fusion_weight.grad[1:].abs().min().item(),
    "MLP w column": m.anatomical_prior_generator.mlp[0].weight.grad[:, -1].abs().max().item(),
    "encoder1.conv1 new ch": m.encoder1.layer.conv1.conv.weight.grad[:, 1:].abs().amax(dim=(0, 2, 3, 4)).min().item(),
    "encoder1.conv3 new ch": m.encoder1.layer.conv3.conv.weight.grad[:, 1:].abs().amax(dim=(0, 2, 3, 4)).min().item(),
}
for name, g in grads.items():
    check(f"3. non-zero gradient: {name}", g > 0, f"min |grad| {g:.2e}")
del m, logits, loss; torch.cuda.empty_cache()

# --- 4: gate = 0 removes the prior from the ViT path ---------------------------------------------
m = AtlasAwareViTUNETR(**MODEL_KW, conf_gating=True).to(device).eval()
load_baseline_weights(m, BASELINE_CKPT)
with torch.no_grad():
    for b in m.vit.blocks:
        b.cross_gate.zero_()
    a = m(img_t, prob_t, dist_t, conf_t)
    b_ = m(img_t, torch.rand_like(prob_t), torch.rand_like(dist_t), torch.rand_like(conf_t))
check("4. all gates 0: output independent of the prior", torch.equal(a, b_))
del m; torch.cuda.empty_cache()

# --- 5: output-fusion formula --------------------------------------------------------------------
m = AtlasAwareViTUNETR(**MODEL_KW, output_fusion=True).to(device).eval()
load_baseline_weights(m, BASELINE_CKPT)
gamma = torch.linspace(-1, 1, NUM_CLASSES, device=device)
with torch.no_grad():
    l0 = m(img_t, prob_t, dist_t, conf_t)
    m.output_fusion_weight.copy_(gamma)
    l1 = m(img_t, prob_t, dist_t, conf_t)
    l_drop = m(img_t, prob_t, dist_t, torch.zeros_like(conf_t))
expected = gamma.view(1, -1, 1, 1, 1) * conf_t * torch.log(prob_t + LOG_PRIOR_EPS)
err = (l1 - l0 - expected).abs().max().item()
check("5. fusion adds exactly γ·w·log(P+ε)", err < 1e-4, f"max error {err:.1e}")
check("5. fusion is inactive where w = 0", torch.allclose(l_drop, l0, atol=1e-5))
del m; torch.cuda.empty_cache()

# --- 6: prior dropout ----------------------------------------------------------------------------
smp = {"prob": prob_t[0].cpu(), "dist": dist_t[0].cpu(), "conf": conf_t[0].cpu()}
dropped = PriorDropoutd(prob=1.0)(smp)
check("6. dropout p=1 -> P uniform, D=1, w=0",
      torch.allclose(dropped["prob"], torch.full_like(smp["prob"], 1 / NUM_CLASSES))
      and bool((dropped["dist"] == 1).all()) and bool((dropped["conf"] == 0).all()))
kept = PriorDropoutd(prob=0.0)(smp)
check("6. dropout p=0 is the identity", all(torch.equal(kept[k], smp[k]) for k in smp))
pdrop = PriorDropoutd(prob=0.3); pdrop.set_random_state(seed=0)
tiny = {"prob": torch.rand(NUM_CLASSES, 2, 2, 2), "dist": torch.rand(NUM_CLASSES - 1, 2, 2, 2), "conf": torch.rand(1, 2, 2, 2)}
rate = sum(bool((pdrop(tiny)["conf"] == 0).all()) for _ in range(2000)) / 2000
check("6. dropout p=0.3 fires ~30% of samples", abs(rate - 0.3) < 0.03, f"observed {rate:.3f}")

# --- 7: consistency loss -------------------------------------------------------------------------
pp = torch.rand(2, NUM_CLASSES, 4, 4, 4) + 0.01; pp = pp / pp.sum(1, keepdim=True)
ww = torch.rand(2, 1, 4, 4, 4)
rnd = atlas_consistency_loss(torch.randn(2, NUM_CLASSES, 4, 4, 4), pp, ww).item()
same = atlas_consistency_loss(torch.log(pp), pp, ww).item()
zero_w = atlas_consistency_loss(torch.randn(2, NUM_CLASSES, 4, 4, 4), pp, torch.zeros_like(ww)).item()
check("7. KL >= 0 for random logits", rnd >= 0, f"{rnd:.4f}")
check("7. KL ~ 0 when softmax(logits) = P", abs(same) < 1e-5, f"{same:.1e}")
check("7. KL = 0 when w = 0", zero_w == 0.0)
ramp = [round(consistency_lambda(e, 0.1), 3) for e in (0, 5, 10, 20)]
check("7. λ ramp 0 -> 0.1 over 10 epochs", ramp == [0.0, 0.05, 0.1, 0.1], f"epochs 0/5/10/20: {ramp}")

test_table = pd.DataFrame(results)
n_fail = int((test_table["result"] == "FAIL").sum())
print(f"\n{len(test_table) - n_fail}/{len(test_table)} checks passed")
assert n_fail == 0, test_table[test_table["result"] == "FAIL"]
test_table

<!-- v2-model-section -->
### Real-data smoke run (demonstration only, not a result)

Same setup as the section 8 smoke test (3 cached training targets, 2 validation cases, AdamW 1e-4, fp16), but with
the redesigned model, all flags on, prior dropout 0.3 and the consistency loss at its full weight 0.1 (no ramp, so
it is exercised). The GradScaler starts at 2^12 instead of the default 2^16, which overflows and skips the first
few steps (for the epoch-101 model as well), so a 3-step demo would otherwise make no update at all. Checks only that the whole pipeline trains end to end on this 4 GB GPU and that at step 0 it
scores the same as epoch-101. Three steps say nothing about whether the redesign helps; that is the server run
(experiment flow, steps 1 to 5 in `new_model_plan.md`). If the section 8 `model` is on the GPU it is moved to the CPU
first to free memory. Skipped in a full run (`FULL_RUN = True`), where section 10 does the real thing.

In [ ]:
#v2-cell smoke
from monai.data import DataLoader, Dataset
from monai.losses import DiceCELoss

RUN_V2_SMOKE = globals().get("RUN_V2_SMOKE", True)   # set in the configuration cell
V2_STEPS = 3

cached_val, _ = attach_atlas_priors(val_cases, "tps")
tf_v2_val = build_v2_transforms(train=False)
if RUN_V2_SMOKE and cached_val:
    if "model" in globals() and isinstance(globals()["model"], nn.Module):
        model.cpu()
    torch.cuda.empty_cache()
    train_cached, _ = attach_atlas_priors(train_targets, "tps")
    v_loader = DataLoader(Dataset(cached_val[:2], tf_v2_val), batch_size=1)
    tf_v2_train = build_v2_transforms(train=True, prior_dropout=0.3)
    tf_v2_train.set_random_state(seed=0)
    t_loader = DataLoader(Dataset(train_cached[:V2_STEPS], tf_v2_train), batch_size=1, shuffle=True)
    use_amp = device.type == "cuda"

    v2 = AtlasAwareViTUNETR(**MODEL_KW, **ALL_FLAGS).to(device)
    load_baseline_weights(v2, BASELINE_CKPT)
    d_init, _ = v2_validate(v2, v_loader, device, amp=use_amp)

    opt = torch.optim.AdamW(v2.parameters(), lr=1e-4, weight_decay=1e-5)
    # Default init_scale 65536 overflows on the first steps and the scaler skips them (it does so for the epoch-101
    # model too); harmless over an epoch, but it would turn a 3-step demo into 0 updates.
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp, init_scale=2.0 ** 12)
    t0 = time.time()
    v2_train_steps(v2, t_loader, opt, DiceCELoss(to_onehot_y=False, softmax=True), device, scaler=scaler,
                   amp=use_amp, max_steps=V2_STEPS, consistency_weight=0.1, ramp_epochs=0)
    d_after, per_after = v2_validate(v2, v_loader, device, amp=use_amp)
    mem = f", peak GPU {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB" if use_amp else ""
    print(f"redesigned model, {len(v_loader)} val cases with atlas priors: Dice {d_init:.3f} at init (= epoch-101) "
          f"-> {d_after:.3f} after {V2_STEPS} steps ({time.time() - t0:.0f}s{mem})")
    print("per organ:", {f"o{k}": None if s is None else round(s, 2) for k, s in zip(ORGAN_IDS, per_after)})
    newp = v2.new_parameters()
    print("cross gates g_i:", [round(x, 5) for x in newp["cross_gates"].tolist()])
    print("fusion γ_c     :", [round(x, 5) for x in newp["gamma"].tolist()])
    del v2, opt; torch.cuda.empty_cache()
else:
    print("skipped (needs RUN_V2_SMOKE and cached registrations)")

<!-- v2-model-section -->
## 10. Full run: the redesigned model on every target (server)

With `FULL_RUN = True` in the configuration cell, this section is the main experiment. It

1. **registers** the 10 atlases onto every non-atlas training target (534) and every validation case (136) in
   `REG_SHARDS` parallel processes. Targets already in `cache/` are skipped, so re-running after an interruption
   continues where it stopped; a target that fails is logged and skipped (`--continue-on-error`) instead of stopping
   its shard, and a shard that crashes (e.g. GPU out of memory) is retried once in a single process;
2. **benchmarks** a few training steps on this machine and prints the estimated training time;
3. **trains `AtlasAwareViTUNETR` from scratch** with every change switched on (C, D, E, prior dropout B,
   consistency loss F) on all registered training targets (`scripts/train_prototype_v2.py --full-data`, which runs
   the model code of section 9), validating on all 136 validation cases every epoch. Re-running the cell resumes
   from `checkpoint_last.pth`;
4. **plots** the curves against the atlas-only reference.

From scratch, not from epoch 101: those weights learned to copy a ground-truth prior (section 8), and C and D change
tensor shapes anyway. The micro-batch is chosen from GPU memory (1 on 4 GB, 2 from 10 GB, 4 from 20 GB); gradient
accumulation brings it to `V2_EFFECTIVE_BATCH`. The network has only instance and layer norm, so this is the same as
a real batch of that size.

**Cost** (measured on the 4 GB laptop; a server is faster): registration ~3.6 min per target per process, i.e. about
40 h for 670 targets in one process; training ~1.5 s per sample, i.e. ~14 min per epoch over 534 targets. **Disk:**
~1 MB of warped labels per target in `cache/warped_labels/tps`, and ~13 MB per case of preprocessed samples in
`cache/persistent_v2_u8` (~9 GB).

**Local chunk run, for reference** (24 train / 6 validation cases, 60 epochs, 42 min): best validation Dice 0.168
against 0.350 for the atlas vote alone, and 0.054 with the prior removed. That run predates a fix in the sample cache:
it dropped each volume's affine, so D was computed in voxel units instead of mm for *training* samples while
validation samples were correct. The full run uses the fixed cache, which was checked to give bit-identical samples
to the uncached pipeline.

In [ ]:
#v2-cell full-registration
from data.splits import warped_label_path

V2_DIR = REPO_ROOT / "experiments" / V2_EXPERIMENT
REG_JSON_DIR = REPO_ROOT / "cache" / "registrations" / "tps"
full_train, full_val = split_with_atlases_removed(DATA_DIR)


def registered(case):
    return os.path.exists(warped_label_path(case, "tps")) and (REG_JSON_DIR / f"{case_name(case['image'])}.json").exists()


def registration_status():
    return {"train": sum(map(registered, full_train)), "val": sum(map(registered, full_val))}


def run_registration_shards(split, shards, log_dir, extra=()):
    # One process per shard, output to log files; prints progress every 10 minutes.
    log_dir.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    procs = []
    for i in range(shards):
        cmd = [sys.executable, "scripts/register_targets.py", "--data-dir", str(DATA_DIR), "--split", split,
               "--num-shards", str(shards), "--shard-index", str(i), "--continue-on-error", *extra]
        log_f = open(log_dir / f"{split}_shard{i}_of{shards}.log", "a", encoding="utf-8")
        procs.append((subprocess.Popen(cmd, cwd=REPO_ROOT, env=env, stdout=log_f, stderr=subprocess.STDOUT), log_f))
    t0 = last = time.time()
    while any(p.poll() is None for p, _ in procs):
        time.sleep(30)
        if time.time() - last >= 600:
            last = time.time()
            s = registration_status()
            print(f"  [{(time.time() - t0) / 60:4.0f} min] registered: train {s['train']}/{len(full_train)}, "
                  f"val {s['val']}/{len(full_val)}", flush=True)
    for _, log_f in procs:
        log_f.close()
    return [p.returncode for p, _ in procs]


past = [json.loads(p.read_text(encoding="utf-8"))["elapsed_s"] for p in REG_JSON_DIR.glob("*.json")
        if not p.name.startswith("_")] if REG_JSON_DIR.exists() else []
s_per_target = float(np.median(past)) if past else 218.0
st = registration_status()
todo = len(full_train) + len(full_val) - st["train"] - st["val"]
print(f"targets: {len(full_train)} train + {len(full_val)} val | already registered: train {st['train']}, val {st['val']}")
print(f"to register: {todo} -> about {todo * s_per_target / REG_SHARDS / 3600:.1f} h with {REG_SHARDS} processes at "
      f"{s_per_target:.0f} s per target ({'median of ' + str(len(past)) + ' runs here' if past else 'laptop figure'}; "
      f"processes share one GPU, so the real speed-up is usually below {REG_SHARDS}x)")

if RUN_FULL_REGISTRATION and todo:
    for split, extra in (("val", ("--eval-gt",)), ("train", ())):
        t0 = time.time()
        codes = run_registration_shards(split, REG_SHARDS, V2_DIR / "registration_logs", extra)
        print(f"{split}: {REG_SHARDS} process(es) finished in {(time.time() - t0) / 3600:.2f} h, exit codes {codes}")
        if any(codes):
            print(f"{split}: a process crashed (see registration_logs); retrying in one process, cached targets are skipped")
            run_registration_shards(split, 1, V2_DIR / "registration_logs", extra)

st = registration_status()
unregistered = [case_name(c["image"]) for c in full_train + full_val if not registered(c)]
print(f"registered now: train {st['train']}/{len(full_train)}, val {st['val']}/{len(full_val)}")
if unregistered:
    print(f"{len(unregistered)} target(s) without registration (training skips them):", unregistered[:20])

In [ ]:
#v2-cell full-estimate
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30 if device.type == "cuda" else 0.0
MICRO_BATCH = 4 if gpu_gib >= 20 else 2 if gpu_gib >= 10 else 1
ACCUM = max(1, V2_EFFECTIVE_BATCH // MICRO_BATCH)
NUM_WORKERS = V2_NUM_WORKERS
if NUM_WORKERS is None:
    # Each worker re-imports torch/monai/SimpleITK and runs the full resample pipeline, which costs real RAM
    # (observed ~1.5-2 GB/worker here); cpu_count alone over-subscribes a laptop. Scale by free RAM too, floor 1.
    try:
        import psutil
        free_gib = psutil.virtual_memory().available / 2**30
    except Exception:
        free_gib = 8.0  # psutil not installed: assume a laptop-sized machine
    NUM_WORKERS = max(1, min(int(free_gib // 2), (os.cpu_count() or 4) // 4, 8))
V2_ARGS = ["scripts/train_prototype_v2.py", "--full-data", "--data-dir", str(DATA_DIR), "--experiment", V2_EXPERIMENT,
           "--epochs", str(V2_EPOCHS), "--micro-batch", str(MICRO_BATCH), "--accum", str(ACCUM),
           "--num-workers", str(NUM_WORKERS), "--amp", "--allow-missing"]
print(f"GPU {gpu_gib:.0f} GiB -> micro-batch {MICRO_BATCH} x accumulation {ACCUM} = effective batch "
      f"{MICRO_BATCH * ACCUM} | {NUM_WORKERS} data workers")

st = registration_status()
if RUN_V2_BENCH and st["train"] and st["val"]:
    run_script([*V2_ARGS, "--bench", "--run-name", "bench"])
    b = json.loads((V2_DIR / "bench" / "bench.json").read_text(encoding="utf-8"))
    gpu_s = b["step_s_per_micro_batch"] / MICRO_BATCH
    data_s = b["data_s_per_sample_main_process"] / NUM_WORKERS
    per_sample = max(gpu_s, data_s)
    epoch_s = st["train"] * per_sample + 1.2 * st["val"] * b["val_s_per_case"]   # +20 %: no-prior check every 5th epoch
    cache_s = (st["train"] + st["val"]) * b["first_pass_s_per_case"] / NUM_WORKERS
    print(f"\nestimate: {per_sample:.2f} s per training sample ({'GPU' if gpu_s >= data_s else 'data loading'}-bound), "
          f"~{epoch_s / 60:.1f} min per epoch -> {V2_EPOCHS} epochs ~{V2_EPOCHS * epoch_s / 3600:.1f} h, "
          f"plus ~{cache_s / 60:.0f} min in epoch 1 to build the sample cache (peak GPU {b['peak_gpu_gib']:.1f} GiB)")
else:
    print("benchmark skipped (RUN_V2_BENCH = False, or no registered targets yet)")

In [ ]:
#v2-cell full-train
run_dir = V2_DIR / V2_RUN_NAME
train_args = [*V2_ARGS, "--run-name", V2_RUN_NAME]
if (run_dir / "checkpoint_last.pth").exists():
    train_args += ["--resume", str(run_dir / "checkpoint_last.pth")]
    print(f"resuming {run_dir}")
elif run_dir.exists():
    stale = run_dir.with_name(f"{V2_RUN_NAME}_incomplete_{time.strftime('%Y%m%d_%H%M%S')}")
    run_dir.rename(stale)
    print(f"{run_dir.name} has no checkpoint (it stopped during epoch 1); moved aside to {stale.name}")

if RUN_V2_FULL_TRAIN:
    st = registration_status()
    if st["train"] < len(full_train) or st["val"] < len(full_val):
        print(f"WARNING: only the registered targets are used: train {st['train']}/{len(full_train)}, "
              f"val {st['val']}/{len(full_val)}")
    if globals().get("model") is not None:
        model.cpu()          # the section 8 network would otherwise hold GPU memory in this kernel
    torch.cuda.empty_cache()
    run_script(train_args)
else:
    print("skipped (RUN_V2_FULL_TRAIN = False)")

In [ ]:
#v2-cell full-results
run_dir = V2_DIR / V2_RUN_NAME
metrics_path = run_dir / "metrics.csv"
if metrics_path.exists() and len(pd.read_csv(metrics_path)):
    m = pd.read_csv(metrics_path)
    ref_m = re.search(r"majority vote on the validation chunk \(96\^3 grid\): Dice ([0-9.]+)",
                      (run_dir / "training.log").read_text(encoding="utf-8"))
    ref = float(ref_m.group(1)) if ref_m else None

    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    ax[0].plot(m.epoch, m.train_dice_ce, label="DiceCE")
    ax[0].plot(m.epoch, m.train_kl, label="KL(P || prediction), before x lambda")
    ax[0].set_title("training loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(m.epoch, m.val_dice, label="validation Dice, atlas prior")
    np_rows = m[m.val_dice_no_prior.notna()]
    ax[1].plot(np_rows.epoch, np_rows.val_dice_no_prior, "o--", label="same, prior removed")
    if ref is not None:
        ax[1].axhline(ref, color="gray", ls=":", label=f"atlas vote alone ({ref:.3f})")
    ax[1].set_title(f"validation ({json.loads((run_dir / 'config.json').read_text())['num_val']} cases)")
    ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[2].plot(m.epoch, m.gate_mean, label="mean cross-attention gate g")
    ax[2].plot(m.epoch, m.gamma_mean, label="mean output-fusion weight gamma (organs)")
    ax[2].set_title("how much the model trusts the prior"); ax[2].set_xlabel("epoch"); ax[2].legend()
    plt.tight_layout(); plt.show()

    best = m.loc[m.val_dice.idxmax()]
    print(f"{len(m)} epochs | best epoch {int(best.epoch)}: validation Dice {best.val_dice:.4f}"
          + (f" | atlas vote alone {ref:.4f}" if ref is not None else ""))
    organ_cols = [f"o{k}" for k in ORGAN_IDS]
    display(pd.DataFrame({"best epoch": best[organ_cols], "last epoch": m.iloc[-1][organ_cols]}).astype(float).round(3).T)
else:
    print(f"no full-run metrics yet in {run_dir}")

## 11. What is done, found, and still open

**Bugs and problems found along the way** (all in `research_audit.md`)
1. The prototype's P(v) and D(v) came from each case's own ground truth: leakage in training and validation.
2. The distance map was inverted (fixed; `legacy_buggy_distance=True` reproduces the old behaviour).
3. `RandRotated(range_x=(-15, 15))` is in **radians**, so the prototype trained with arbitrary rotations rather than
   ±15°. Kept as the default so Experiment 1 changes only the prior; `--rotate-range-deg 15` is the corrected ablation.
4. The label is resized with PyTorch `nearest`, which sits about 0.5 to 0.9 voxel off the CT (resized with `area`).
   It affects the prototype too; documented, not changed.
5. `requirements.txt` originally resolved to a CPU-only torch; it now pins the CUDA build.
6. A sign error in my TPS smoothing term (`K + λI` instead of `K − λI`) made every fit erratic and had hidden the TPS's
   benefit. Found while validating this notebook; atlas-only Dice went from about 0.32 to 0.45 to 0.48 after the fix.
7. The plan's "load epoch-101 with `strict=False`" cannot work for the redesign (C and D change tensor shapes);
   `load_baseline_weights` copies into the leading slice instead, and the full run trains from scratch anyway.
8. MONAI's `PersistentDataset` drops MetaTensor affines unless `track_meta=True`, which made `AtlasPriorsd` compute
   D in voxel units for cached *training* samples (validation was unaffected). Found in the local chunk run, fixed in
   `scripts/train_prototype_v2.py` and checked bit-identical against the uncached pipeline.

**Still open**
- **Registration quality**: atlas-only Dice is 0.45 to 0.48 on two validation cases; tune the golden-transform λ,
  reliability threshold, TPS λ, RANSAC threshold and patch size on non-atlas *training* targets (never on
  validation). Small organs stay weak.
- **Does the model even see registration detail?** The priors enter only through 16³-patch pooling. A sensitivity test
  (shift the priors by known amounts and watch Dice) would show whether ~10 mm registration gains can matter at all.
- **Experiment 1 at scale**: register all targets and train on a server; compare against a no-prior reference model
  and against the epoch-101 baseline run with ground-truth priors.
- **Doctor-segmented atlases**: replace the 10 temporary atlases when available (one file plus cache rebuild).
- **Redesigned model at scale** (section 10): the local chunk (24 targets) reached 0.168 against 0.350 for the atlas
  vote alone, i.e. it does not yet beat its own prior. The full run says whether that is a data-size effect.
- **Converting the prototype**: once accepted, section 9's code moves into `models/`, `training/` and
  `scripts/train.py`, and the checks into `tests/`.
- **Experiments 3 and 4 are not implemented**: CT edge loss, and an HU loss only if it can be justified, since
  intensities are rescaled to [0, 1]. (Experiment 2, the confidence-weighted consistency loss, is change F.)
- **Not from the brief yet**: the `run.py` orchestrator and the `config/` files.